In [352]:
import pickle

with open("data/train_processed.pkl", "rb") as f:
    train_processed = pickle.load(f)

with open("data/val_processed.pkl", "rb") as f:
    val_processed = pickle.load(f)

with open("data/test_processed.pkl", "rb") as f:
    test_processed = pickle.load(f)

with open("data/word2idx_2.pkl", "rb") as f:
    word2idx_2 = pickle.load(f)

with open("data/idx2word_2.pkl", "rb") as f:
    idx2word_2 = pickle.load(f)

In [353]:
import pandas as pd

qa_df = pd.read_pickle("sample_qa.pkl")

In [354]:
import torch
from torch.utils.data import Dataset, DataLoader

import numpy as np

from scipy.io import loadmat

In [355]:
from scipy.io import loadmat

import torch
from torch.utils.data import Dataset

class SARVQADataset(Dataset):

    def __init__(self, data):

        self.data = data

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        sample = self.data[idx]

        filepath = sample["filepath"]

        mat = loadmat(filepath)

        complex_img = mat["complex_img"]

        image = torch.tensor(
            complex_img,
            dtype=torch.cfloat
        )

        question = torch.tensor(
            sample["question_tokens"],
            dtype=torch.long
        )

        answer = torch.tensor(
            sample["answer"],
            dtype=torch.long
        )

        return image, question, answer

In [356]:
train_dataset = SARVQADataset(train_processed)

image, question, answer = train_dataset[0]

print(image.shape)
print(image.dtype)

print(question.shape)
print(answer)

torch.Size([128, 128])
torch.complex64
torch.Size([5])
tensor(1)


In [357]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [358]:
val_dataset = SARVQADataset(
    val_processed
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

Building a complex 2D convolution function

In [359]:
import torch
import torch.nn as nn

class ComplexConv2d(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        padding=0,
        stride=1
    ):

        super().__init__()

        self.real_conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            stride=stride
        )

        self.imag_conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            stride=stride
        )

    def forward(self, x):

        xr = x.real
        xi = x.imag

        real = (
            self.real_conv(xr)
            -
            self.imag_conv(xi)
        )

        imag = (
            self.real_conv(xi)
            +
            self.imag_conv(xr)
        )

        return torch.complex(
            real,
            imag
        )

In [360]:
conv = ComplexConv2d(
    1,
    16,
    kernel_size=3,
    padding=1
)

x = torch.randn(
    2,
    1,
    128,
    128,
    dtype=torch.cfloat
)

y = conv(x)

print(y.shape)
print(y.dtype)

torch.Size([2, 16, 128, 128])
torch.complex64


Building the CVNN Baseline

In [361]:
import torch
import torch.nn as nn

class QuestionEncoder(nn.Module):

    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, questions):

        x = self.embedding(questions)

        _, (h, _) = self.lstm(x)

        h_forward = h[-2]
        h_backward = h[-1]

        return torch.cat(
            [h_forward, h_backward],
            dim=1
        )

In [362]:
import torchcvnn.nn as c_nn
import complextorch.nn as ct_nn

print(dir(c_nn))

for x in dir(c_nn):
    print(x)

['AdaptiveAvgPool2d', 'AvgPool2d', 'BatchNorm1d', 'BatchNorm2d', 'CCELU', 'CELU', 'CGELU', 'CPReLU', 'CReLU', 'CSigmoid', 'CTanh', 'Cardioid', 'ComplexMSELoss', 'ConvTranspose2d', 'Dropout', 'Dropout2d', 'LayerNorm', 'MaxPool2d', 'Mod', 'MultiheadAttention', 'RMSNorm', 'Transformer', 'TransformerDecoder', 'TransformerDecoderLayer', 'TransformerEncoder', 'TransformerEncoderLayer', 'Upsample', 'ViT', 'ViTLayer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'activation', 'batchnorm', 'conv', 'dropout', 'functional', 'init', 'initialization', 'loss', 'modReLU', 'modules', 'normalization', 'pooling', 'transformer', 'upsampling', 'vit', 'zAbsReLU', 'zLeakyReLU', 'zReLU']
AdaptiveAvgPool2d
AvgPool2d
BatchNorm1d
BatchNorm2d
CCELU
CELU
CGELU
CPReLU
CReLU
CSigmoid
CTanh
Cardioid
ComplexMSELoss
ConvTranspose2d
Dropout
Dropout2d
LayerNorm
MaxPool2d
Mod
MultiheadAttention
RMSNorm
Transformer
TransformerDecoder
TransformerDecod

Creating the encoder

In [363]:
class ComplexSARImageEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = ct_nn.Conv2d(
            1,
            16,
            kernel_size=3,
            padding=1
        )

        self.act1 = c_nn.CReLU()

        self.pool1 = c_nn.MaxPool2d(
            kernel_size=2)

        self.conv2 = ct_nn.Conv2d(
            16,
            32,
            kernel_size=3,
            padding=1
        )

        self.act2 = c_nn.CReLU()

        self.pool2 = c_nn.MaxPool2d(
            kernel_size=2)

        self.conv3 = ct_nn.Conv2d(
            32,
            32,
            kernel_size=3,
            padding=1
        )

        self.act3 = c_nn.CReLU()

        self.pool3 = c_nn.MaxPool2d(
            kernel_size=2)

        self.avgpool = c_nn.AdaptiveAvgPool2d(
            (1, 1)
        )

    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.act3(x)
        x = self.pool3(x)

        x = self.avgpool(x)

        x = x.flatten(1)

        return x

In [364]:
encoder = ComplexSARImageEncoder()

images, questions, answers = next(
    iter(train_loader)
)

out = encoder(images)

print(out.shape)
print(out.dtype)

torch.Size([32, 32])
torch.complex64


In [365]:
class ComplexSemanticEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(
            64,
            48
        ).to(torch.cfloat)

        self.act1 = c_nn.CReLU()

        self.fc2 = nn.Linear(
            48,
            32
        ).to(torch.cfloat)

    def forward(self, x):

        x = self.fc1(x)

        x = self.act1(x)

        x = self.fc2(x)

        return x

In [366]:
def complex_awgn(
    x,
    snr_db
):

    signal_power = (
        torch.abs(x) ** 2
    ).mean()

    snr_linear = (
        10 ** (snr_db / 10)
    )

    noise_power = (
        signal_power
        / snr_linear
    )

    noise_real = (
        torch.randn_like(
            x.real
        )
        * torch.sqrt(
            noise_power / 2
        )
    )

    noise_imag = (
        torch.randn_like(
            x.imag
        )
        * torch.sqrt(
            noise_power / 2
        )
    )

    noise = torch.complex(
        noise_real,
        noise_imag
    )

    return x + noise

In [367]:
class ComplexSemanticDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(
            32,
            48
        ).to(torch.cfloat)

        self.act1 = c_nn.CReLU()

        self.fc2 = nn.Linear(
            48,
            32
        ).to(torch.cfloat)

    def forward(self, x):

        x = self.fc1(x)

        x = self.act1(x)

        x = self.fc2(x)

        return x

Need to keep dimensional information the same, so we are using only 32 output from the decoder and converting that to real numbers, instead of 64->128, which will increase the dimensions

In [368]:
def complex_to_real(x):

    return torch.cat(
        [
            x.real,
            x.imag
        ],
        dim=1
    )

In [369]:
class VQAClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                192,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                2
            )
        )

    def forward(self, x):

        return self.net(x)

In [370]:
class CVNNVQA(nn.Module):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.image_encoder = (
            ComplexSARImageEncoder()
        )

        self.encoder = (
            ComplexSemanticEncoder()
        )

        self.decoder = (
            ComplexSemanticDecoder()
        )

        self.question_encoder = (
            QuestionEncoder(
                vocab_size
            )
        )

        self.classifier = (
            VQAClassifier()
        )

    def forward(
        self,
        images,
        questions
    ):

        image_features = (
            self.image_encoder(
                images
            )
        )

        # compressed = (
        #     self.encoder(
        #         image_features
        #     )
        # )

        # received = complex_awgn(
        #     compressed,
        #     snr_db=10
        # )

        # reconstructed = (
        #     self.decoder(
        #         received
        #     )
        # )

        # image_vector = complex_to_real(reconstructed)

        image_vector = torch.cat([image_features.real, image_features.imag], dim=1)
        
        question_vector = (
            self.question_encoder(
                questions
            )
        ) 

        fused = torch.cat(
            [
                image_vector,
                question_vector
            ],
            dim=1
        )

        logits = (
            self.classifier(
                fused
            )
        )

        return logits

In [371]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [372]:
model = CVNNVQA(
    vocab_size=len(word2idx_2)
).to(device)

images, questions, answers = next(
    iter(train_loader)
)

images = images.to(device)
questions = questions.to(device)

logits = model(
    images,
    questions
)

print(logits.shape)

torch.Size([32, 2])


C:\Users\Shourya\AppData\Local\Temp\ipykernel_20824\4103612452.py:10: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_20824\4103612452.py:17: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)


In [373]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [374]:
num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, questions, answers in train_loader:

        images = images.to(device)
        questions = questions.to(device)
        answers = answers.to(device)

        optimizer.zero_grad()

        logits = model(
            images,
            questions
        )

        loss = criterion(
            logits,
            answers
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == answers
        ).sum().item()

        total += answers.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {total_loss/len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )



Epoch [1/20] Loss: 0.5841 Accuracy: 67.82%
Epoch [2/20] Loss: 0.5702 Accuracy: 69.05%
Epoch [3/20] Loss: 0.5678 Accuracy: 69.13%


KeyboardInterrupt: 

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, questions, answers in val_loader:

        images = images.to(device)
        questions = questions.to(device)
        answers = answers.to(device)

        logits = model(
            images,
            questions
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == answers
        ).sum().item()

        total += answers.size(0)

val_accuracy = 100 * correct / total

print(
    f"Validation Accuracy: {val_accuracy:.2f}%"
)

Validation Accuracy: 68.96%
